# 04 — Predict Upcoming NBA Games

In [18]:
import pandas as pd
import numpy as np
import joblib
import requests
from pathlib import Path
import os

In [19]:
rf = joblib.load("../models/random_forest.pkl")
feature_cols = joblib.load("../models/model_features.pkl")

team_games = pd.read_csv("../data/modelling/team_games.csv")
team_games["game_date"] = pd.to_datetime(team_games["game_date"], errors="coerce")

team_games.head()

,season_id,team_id,team_name,game_id,game_date,min,fgm,fga,fg_pct,fg3m,...,video_available_roll_5,video_available_roll_10,win_streak_1,win_streak_3,win_streak_5,win_streak_10,prev_game_date,rest_days,is_b2b,rest_3plus
0,22025,1610612737,Atlanta Hawks,22500082,2025-10-22,240,38,90,0.422,10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0
1,22025,1610612737,Atlanta Hawks,22500090,2025-10-24,240,41,85,0.482,8,...,NaN,NaN,0.0,NaN,NaN,NaN,2025-10-22,2.0,0,0
2,22025,1610612737,Atlanta Hawks,22500101,2025-10-25,240,35,85,0.412,16,...,NaN,NaN,1.0,NaN,NaN,NaN,2025-10-24,1.0,1,0
3,22025,1610612737,Atlanta Hawks,22500115,2025-10-27,240,49,98,0.500,11,...,NaN,NaN,0.0,1.0,NaN,NaN,2025-10-25,2.0,0,0
4,22025,1610612737,Atlanta Hawks,22500130,2025-10-29,240,44,93,0.473,13,...,NaN,NaN,0.0,1.0,NaN,NaN,2025-10-27,2.0,0,0


In [20]:
SCHEDULE_URL = "https://cdn.nba.com/static/json/staticData/scheduleLeagueV2.json"
data = requests.get(SCHEDULE_URL).json()

rows = []
for gdate in data["leagueSchedule"]["gameDates"]:
    date = gdate["gameDate"]
    for g in gdate["games"]:
        rows.append({
            "Date": date,
            "Home": g["homeTeam"]["teamName"],
            "Visitor": g["awayTeam"]["teamName"]
        })

schedule = pd.DataFrame(rows)
schedule["Date"] = pd.to_datetime(schedule["Date"], errors="coerce")

today = pd.Timestamp.today().normalize()
upcoming = schedule[schedule["Date"] >= today].sort_values("Date").reset_index(drop=True)

upcoming.head()

,Date,Home,Visitor
0,2026-03-12,Pistons,76ers
1,2026-03-12,Pacers,Suns
2,2026-03-12,Magic,Wizards
3,2026-03-12,Hawks,Nets
4,2026-03-12,Heat,Bucks


In [21]:
# mapping nicknames -> full names
NICK_TO_FULL = {
    "Hawks":"Atlanta Hawks","Celtics":"Boston Celtics","Nets":"Brooklyn Nets","Hornets":"Charlotte Hornets",
    "Bulls":"Chicago Bulls","Cavaliers":"Cleveland Cavaliers","Mavericks":"Dallas Mavericks","Nuggets":"Denver Nuggets",
    "Pistons":"Detroit Pistons","Warriors":"Golden State Warriors","Rockets":"Houston Rockets","Pacers":"Indiana Pacers",
    "Clippers":"LA Clippers","LA Clippers":"LA Clippers","Los Angeles Clippers":"LA Clippers","Lakers":"Los Angeles Lakers",
    "Grizzlies":"Memphis Grizzlies","Heat":"Miami Heat","Bucks":"Milwaukee Bucks","Timberwolves":"Minnesota Timberwolves",
    "Pelicans":"New Orleans Pelicans","Knicks":"New York Knicks","Thunder":"Oklahoma City Thunder","Magic":"Orlando Magic",
    "76ers":"Philadelphia 76ers","Suns":"Phoenix Suns","Trail Blazers":"Portland Trail Blazers","Kings":"Sacramento Kings",
    "Spurs":"San Antonio Spurs","Raptors":"Toronto Raptors","Jazz":"Utah Jazz"
}

# Actual team names found in team_games
DATASET_TEAMS = set(team_games["team_name"].dropna().astype(str).unique())

def to_dataset_name(name):
    if not isinstance(name, str) or not name.strip():
        return None
    
    if name in DATASET_TEAMS:
        return name
    
    if name in NICK_TO_FULL:
        full = NICK_TO_FULL[name]
        # dataset sometimes uses "Los Angeles Clippers"
        if full == "LA Clippers" and "LA Clippers" not in DATASET_TEAMS \
           and "Los Angeles Clippers" in DATASET_TEAMS:
            return "Los Angeles Clippers"
        if full in DATASET_TEAMS:
            return full

    # fuzzy match fallback
    for ds in DATASET_TEAMS:
        if name in ds:
            return ds

    return None

In [22]:
def latest_team_features(team_name):
    df = team_games[team_games["team_name"] == team_name].copy()
    if df.empty:
        return None
    df = df.sort_values("game_date")
    latest = df.tail(1).copy()
    latest = latest.drop(columns=["home_away_flag"], errors="ignore")
    return latest

In [23]:
def build_game_features(visitor_sched, home_sched):
    home_name = to_dataset_name(home_sched)
    away_name = to_dataset_name(visitor_sched)

    if home_name is None or away_name is None:
        return None

    home_latest = latest_team_features(home_name)
    away_latest = latest_team_features(away_name)

    if home_latest is None or away_latest is None:
        return None

    home_latest = home_latest.add_prefix("home_")
    away_latest = away_latest.add_prefix("away_")

    row = pd.concat([home_latest, away_latest], axis=1)

    # compute diff columns expected by model
    for c in feature_cols:
        if c.startswith("diff_"):
            base = c.replace("diff_", "")
            h = f"home_{base}"
            a = f"away_{base}"
            row[c] = 0.0
            if h in row.columns and a in row.columns:
                row[c] = pd.to_numeric(row[h], errors="coerce") - pd.to_numeric(row[a], errors="coerce")

    # align to model training columns
    row = row.reindex(columns=feature_cols, fill_value=0)
    return row

In [24]:
pred_rows = []

for _, g in upcoming.iterrows():
    date = g["Date"]
    home = g["Home"]
    visitor = g["Visitor"]

    feats = build_game_features(visitor, home)

    if feats is None:
        continue

    proba_home_win = rf.predict_proba(feats)[0][1]  # probability home team wins
    pred_class = rf.predict(feats)[0]

    pred_rows.append({
        "Date": date.date(),
        "Visitor": visitor,
        "Home": home,
        "Home_Win_Probability (%)": round(proba_home_win * 100, 1),
        "Predicted_Winner": home if pred_class == 1 else visitor
    })

pred_df = pd.DataFrame(pred_rows)
pred_df.head()


,Date,Visitor,Home,Home_Win_Probability (%),Predicted_Winner
0,2026-03-12,76ers,Pistons,92.0,Pistons
1,2026-03-12,Suns,Pacers,44.8,Suns
2,2026-03-12,Wizards,Magic,91.2,Magic
3,2026-03-12,Nets,Hawks,93.0,Hawks
4,2026-03-12,Bucks,Heat,91.5,Heat


In [25]:
out_dir = Path("../outputs/predictions")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "upcoming_predictions.csv"
pred_df.to_csv(out_path, index=False)

print(f"Saved predictions → {out_path} ({len(pred_df)} games)")

Saved predictions → ..\outputs\predictions\upcoming_predictions.csv (250 games)
